# 14 Beats

The beats view is a clip launcher in the session-view style: tracks are rows, scenes are columns, and each slot holds a step pattern. Launching a slot starts it on the next bar of the shared session clock, so everything you launch stays phase-locked; launching a scene fills its whole column. A launcher performance plays through the session mixer, which means the timeline can bounce it to audio clips.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

from nbplay import SamplerWidget, SequencerWidget, Session, SynthWidget

session = Session(bpm=120.0, time_signature=(4, 4))
launcher = session.launcher

# Three instrument lanes. Each launcher track plays through its mixer
# channel, so a launcher performance can be bounced by the timeline.
drums = SamplerWidget(pad_count=8)
bass = SynthWidget(oscillator_type="square", frequency=110.0, amplitude=0.4)
keys = SynthWidget(oscillator_type="saw", frequency=440.0, amplitude=0.3)
drum_track = session.add_track("Drums", sound_source=drums)
bass_track = session.add_track("Bass", sound_source=bass)
keys_track = session.add_track("Keys", sound_source=keys)

for scene in ("Intro", "Verse", "Drop"):
    launcher.add_scene(scene)

def steps(notes, velocity=100, every=1):
    return [
        {"note": note, "velocity": velocity, "active": i % every == 0, "duration_ticks": 1, "probability": 100}
        for i, note in enumerate(notes)
    ]

# Patterns can come from step lists, NoteComposers, or SequencerWidgets.
launcher.set_slot(drum_track.mixer_channel, 0, steps([36] * 8, every=2), name="Kick", step_duration=0.5)
launcher.set_slot(drum_track.mixer_channel, 1, steps([36, 38] * 4), name="Kick+Snare", step_duration=0.5)
launcher.set_slot(drum_track.mixer_channel, 2, steps([36, 42, 38, 42] * 4, velocity=110), name="Full kit", step_duration=0.25)

launcher.set_slot(bass_track.mixer_channel, 1, steps([36, 36, 43, 36, 34, 34, 41, 34]), name="Root walk", step_duration=0.5)
launcher.set_slot(bass_track.mixer_channel, 2, steps([36, 48, 36, 48, 34, 46, 34, 46]), name="Octaves", step_duration=0.5)

lead_seq = SequencerWidget(length=16, num_voices=2, step_duration=0.25)
for i, note in enumerate([72, 76, 79, 84, 83, 79, 76, 72, 71, 74, 78, 83, 81, 78, 74, 71]):
    lead_seq.set_step(i, note=note, velocity=90, active=i % 2 == 0, voice=0)
launcher.set_slot(keys_track.mixer_channel, 2, lead_seq, name="Arp")

# Editing: clicking a slot selects it; this sequencer edits whichever slot
# is selected and writes changes straight back into the launcher.
editor = SequencerWidget(length=8, step_duration=0.5)
launcher.bind_slot_editor(editor)
launcher.selected_slot = {"track_index": drum_track.mixer_channel, "scene_index": 0}

# Bounce a performance: arm the lanes (channel taps) and press Rec on the transport.
for track in (drum_track, bass_track, keys_track):
    session.timeline.arm_track(track.mixer_channel, True)

tabs = widgets.Tab([
    widgets.VBox([session.transport, launcher, editor]),
    widgets.VBox([session.timeline, session.mixer]),
    widgets.VBox([drums, bass, keys]),
])
for index, title in enumerate(["Beats", "Tracking + Mixer", "Sources"]):
    tabs.set_title(index, title)
display(tabs)
print(launcher)


Click a slot to launch it (or a scene name to launch the column); the stop buttons at the row ends and **Stop All** take effect on the next bar. Change **Quantize** to launch on beats or immediately. From Python, `launcher.launch(track, scene)`, `launcher.launch_scene(scene)`, `launcher.stop_track(track)`, and `launcher.stop_all()` do the same.

The editor sequencer under the grid shows whichever slot is selected; edit its steps and the slot updates live, even while it plays. To capture a performance, arm lanes on the Tracking tab (instrument lanes default to a mixer-channel tap), press **Rec** on the transport, and perform; the takes land on the timeline as audio clips.
